In [1]:
import numpy as np
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
df = pd.read_csv("Groceries_dataset.csv");
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38765 entries, 0 to 38764
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Member_number    38765 non-null  int64 
 1   Date             38765 non-null  object
 2   itemDescription  38765 non-null  object
dtypes: int64(1), object(2)
memory usage: 908.7+ KB


,Member_number,Date,itemDescription
0,1808,21-07-2015,tropical fruit
1,2552,05-01-2015,whole milk
2,2300,19-09-2015,pip fruit
3,1187,12-12-2015,other vegetables
4,3037,01-02-2015,whole milk


In [3]:
transactions = (
    df
    .groupby(['Member_number', 'Date'])['itemDescription']
    .apply(list)
    .tolist()
)

print("Number of transactions:", len(transactions))
print("Sample transaction:", transactions[1124])


Number of transactions: 14963
Sample transaction: ['tropical fruit', 'cocoa drinks']


In [4]:
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [5]:
frequent_itemsets = apriori(
    df_encoded,
    min_support=0.002,
    use_colnames=True
)

print(frequent_itemsets['itemsets'].apply(len).value_counts())

frequent_itemsets.sort_values(by='support', ascending=False).head(10)

itemsets
2    204
1    126
Name: count, dtype: int64


,support,itemsets
123,0.157923,(whole milk)
78,0.122101,(other vegetables)
93,0.110005,(rolls/buns)
104,0.097106,(soda)
124,0.085879,(yogurt)
94,0.069572,(root vegetables)
116,0.067767,(tropical fruit)
7,0.060683,(bottled water)
99,0.060349,(sausage)
24,0.053131,(citrus fruit)


In [6]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1
)

print(len(rules))

rules.head()

36


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(other vegetables),(berries),0.122101,0.021787,0.002673,0.021894,1.004899,1.0,0.000013,1.000109,0.005553,0.018930,0.000109,0.072297
1,(berries),(other vegetables),0.021787,0.122101,0.002673,0.122699,1.004899,1.0,0.000013,1.000682,0.004984,0.018930,0.000681,0.072297
2,(bottled beer),(sausage),0.045312,0.060349,0.003342,0.073746,1.222000,1.0,0.000607,1.014464,0.190292,0.032658,0.014258,0.064559
3,(sausage),(bottled beer),0.060349,0.045312,0.003342,0.055371,1.222000,1.0,0.000607,1.010649,0.193337,0.032658,0.010537,0.064559
4,(canned beer),(brown bread),0.046916,0.037626,0.002406,0.051282,1.362937,1.0,0.000641,1.014394,0.279398,0.029292,0.014190,0.057613


In [7]:
strong_rules = rules[
    (rules['confidence'] > 0.5) &
    (rules['lift'] > 1.3)
].sort_values(by='lift', ascending=False)

strong_rules.head(10)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
